# PalmSens `.pssession` Folder Workflow

This notebook shows the direct PalmSens route. If your raw data are PalmSens `.pssession` files, you do not need to manually build a structured CSV first. Point ASWIFT at a folder of `.pssession` files and the batch helpers will:

1. read the PalmSens files,
2. extract each SWV trace as a full `voltage` array and full `current` array,
3. preserve metadata such as file, timestamp, frequency, channel, and trace label,
4. preserve relative time so the earliest measurement is `0.0`,
5. fit the traces, and
6. return the same structured raw `results` dataframe that can be saved as JSON and opened in the Streamlit viewer.

The `.pssession` reader requires the optional `pypalmsens` package. This notebook uses the public `pssession_example` folder and downloads the ASWIFT example-data archive when it is not already available locally.


In [ ]:
from pathlib import Path
import sys
import urllib.request
import zipfile

import matplotlib.pyplot as plt
# noinspection PyPackageRequirements
from IPython.display import display

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "src" / "aswift").exists():
        repo_root = candidate
        break
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from aswift import (
    fit_pssession_folder,
    plot_signal_over_time,
    pssession_folder_to_dataframe,
    strip_mp3_suffix_from_pssession_files,
)


## Download Or Point To Example Data

The public archive contains the structured CSV, simple CSV, and PalmSens examples used by these notebooks. Download and extraction are skipped when the `pssession_example` folder is already present.

If files arrive as `.pssession.mp3`, the next cell strips only the final `.mp3` suffix before checking for `.pssession` files.

In [ ]:
EXAMPLE_DATA_URL = "https://github.com/Soh-Lab/aswift/releases/download/v1.0.2/example_data.zip"
examples_dir = repo_root / "examples" if (repo_root / "examples").exists() else Path.cwd().resolve()
bundled_archive = repo_root / "release_assets" / "example_data.zip"
EXAMPLE_DATA_ZIP = bundled_archive if bundled_archive.exists() else examples_dir / "example_data.zip"
PSSESSION_FOLDER = examples_dir / "example_data" / "pssession_example"

if not PSSESSION_FOLDER.exists():
    if not EXAMPLE_DATA_ZIP.exists():
        urllib.request.urlretrieve(EXAMPLE_DATA_URL, EXAMPLE_DATA_ZIP)
    with zipfile.ZipFile(EXAMPLE_DATA_ZIP) as zf:
        zf.extractall(examples_dir)

renamed_files = []
if PSSESSION_FOLDER.exists():
    renamed_files = strip_mp3_suffix_from_pssession_files(PSSESSION_FOLDER)

pssession_files = sorted(PSSESSION_FOLDER.glob("*.pssession")) if PSSESSION_FOLDER.exists() else []
len(renamed_files), renamed_files[:3], len(pssession_files), pssession_files[:3]

## Convert PalmSens Files To The Structured SWV DataFrame

`pssession_folder_to_dataframe` converts a folder of `.pssession` files into the same array-per-trace shape used by notebook 02: one row per voltammogram, with `voltage` and `current` containing full arrays plus metadata columns.

The output includes metadata such as `file`, `timestamp`, `hz`, `channel`, `label`, `voltage`, and `current`. Timestamps are read from PalmSens UTC metadata when available, and `time` is stored in hours relative to the earliest measurement.


In [ ]:
if pssession_files:
    swv_df = pssession_folder_to_dataframe(PSSESSION_FOLDER)
    display(swv_df.head())
else:
    swv_df = None
    print(f"No .pssession files found in {PSSESSION_FOLDER}.")

## Fit All Traces With Threads

`fit_pssession_folder` combines the conversion and fitting steps. It returns both the structured SWV dataframe and the structured results dataframe. The results are ordered by timestamp, frequency, file/sample number, and channel when those columns are available.


In [ ]:
if pssession_files:
    swv_df, results = fit_pssession_folder(
        PSSESSION_FOLDER,
        method="aswift",  # or "poly_linear"
        n_workers=1,
    )
    display(results[["file", "hz", "channel", "peak", "peak_voltage", "success"]].head())
else:
    results = None
    print("No .pssession files found yet.")

## Save Outputs

In [ ]:
OUTPUT_DIR = Path("outputs/pssession_demo")

viewer_process = None
if results is not None and swv_df is not None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    swv_df.to_csv(OUTPUT_DIR / "pssession_long_form.csv", index=False)
    results.to_json(OUTPUT_DIR / "pssession_fit_results.json", orient="records", indent=2)

    import os
    import socket
    import subprocess
    import time


    def available_port(preferred=8501):
        for p in (preferred, 0):
            with socket.socket() as sock:
                try:
                    sock.bind(("localhost", p))
                except OSError:
                    continue
                return sock.getsockname()[1]
        raise RuntimeError("Could not find an available Streamlit port.")


    import aswift.analysis.viewer_cli as aswift_viewer_cli

    viewer_script = Path(str(aswift_viewer_cli.__file__)).resolve()
    results_json = (OUTPUT_DIR / "pssession_fit_results.json").resolve()
    viewer_env = os.environ.copy()
    if src_root.exists():
        viewer_env["PYTHONPATH"] = os.pathsep.join(
            part for part in (str(src_root), viewer_env.get("PYTHONPATH")) if part
        )
    port = available_port()
    viewer_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "--server.headless=true",
            f"--server.port={port}",
            str(viewer_script),
            "--",
            str(results_json),
        ],
        cwd=repo_root,
        env=viewer_env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    time.sleep(1)
    if viewer_process.poll() is not None:
        raise RuntimeError("Streamlit failed to start. Confirm streamlit is installed in this notebook kernel.")
    print(f"Saved results to {results_json}")
    print(f"Streamlit viewer running at http://localhost:{port} with results loaded")
    print(f"Streamlit process id: {viewer_process.pid}")
else:
    print("Fit results are not available yet.")


## Stop The Streamlit Viewer

Run this cell when you are done with the browser tab to stop the background Streamlit process started above.


In [ ]:
import subprocess

viewer_process: subprocess.Popen[bytes] | None  # assigned in the Save Outputs cell above

if viewer_process is not None and viewer_process.poll() is None:
    viewer_process.terminate()
    try:
        viewer_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        viewer_process.kill()
        viewer_process.wait(timeout=5)
    print("Streamlit viewer stopped.")
elif viewer_process is not None:
    print("Streamlit viewer is already stopped.")
else:
    print("No Streamlit viewer process was started in this notebook session.")


## Plot Signal Over Time

In [ ]:
if results is not None:
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    plot_signal_over_time(results, ax=ax)
    plt.show()
else:
    print("Fit results are not available yet.")